# packet_analyze_v2 — Colab 올인원 (pcap 업로드 → 판정)

pcap 파일을 업로드하면 **수집(Suricata/Zeek) → 압축(evidence+드릴다운 번들) → LLM 판정**까지 Colab에서 한 번에 실행한다.

**전제**: 런타임 → 유형 변경 → **T4 GPU** 선택.

흐름: repo clone → 분석도구 설치(Suricata+Zeek) → Ollama+qwen2.5:14b → **pcap 업로드** → analyze.sh → llm_analyze

> Colab엔 docker가 없어서 Zeek는 네이티브로 설치한다. `run_zeek.sh`가 자동 감지(네이티브↔docker).

In [ ]:
# 1. repo clone (최신 코드 push 후 실행할 것)
REPO = "https://github.com/DAADAISMYLIFE/packet_analyze_v2.git"
!rm -rf /content/packet_analyze_v2
!git clone -q $REPO /content/packet_analyze_v2
%cd /content/packet_analyze_v2
!git log --oneline -1

In [ ]:
%%bash
# 2. 분석도구 설치: Suricata(+ET Open 룰), Zeek(네이티브, docker 불필요), tcpdump
set -e
echo '[1/3] Suricata + tcpdump ...'
apt-get -qq update
apt-get -qq install -y suricata tcpdump >/dev/null

echo '[2/3] ET Open 룰 다운로드 (실패 시 보이게) ...'
suricata-update --no-test || echo '⚠️ suricata-update 경고 (계속)'
RULES=/var/lib/suricata/rules/suricata.rules
if [ -s "$RULES" ]; then
  echo "✅ 룰 로드됨: $(wc -l < $RULES) 줄"
else
  echo '❌ 룰 파일 없음/빈 파일 — alert 안 나옴. suricata-update 로그 확인 필요'
fi

echo '[3/3] Zeek (네이티브) ...'
UBU=$(lsb_release -rs)
echo "deb http://download.opensuse.org/repositories/security:/zeek/xUbuntu_${UBU}/ /" > /etc/apt/sources.list.d/zeek.list
curl -fsSL "https://download.opensuse.org/repositories/security:zeek/xUbuntu_${UBU}/Release.key" | gpg --dearmor > /etc/apt/trusted.gpg.d/zeek.gpg
apt-get -qq update
apt-get -qq install -y zeek >/dev/null
ln -sf /opt/zeek/bin/zeek /usr/local/bin/zeek

echo '── 설치 확인 ──'
zeek --version
suricata -V
command -v zeek && command -v suricata

In [ ]:
# 3. Ollama 설치
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 4. Ollama 서버 백그라운드 시작 + 모델 다운로드
import subprocess, os, time

os.makedirs('/content/logs', exist_ok=True)
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/content/logs/ollama.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(3)  # 서버 초기화 대기

print('모델 다운로드 중... (qwen2.5:14b ~9GB, 몇 분 걸림)')
subprocess.run(['ollama', 'pull', 'qwen2.5:14b'], check=True)
print('✅ 완료')

In [ ]:
# 5. Ollama 서버 준비 확인
import urllib.request, time
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2)
        print('✅ Ollama 준비 완료'); break
    except Exception:
        time.sleep(3)
        if i % 5 == 0: print(f'  대기 {i*3}s...')
else:
    print('❌ 타임아웃 — /content/logs/ollama.log 확인')
    !tail -20 /content/logs/ollama.log

In [ ]:
# 6. pcap 업로드
from google.colab import files
import os, shutil

up = files.upload()                 # pcap 파일 선택
fn = list(up.keys())[0]
os.makedirs('pcaps', exist_ok=True)
shutil.move(fn, f'pcaps/{fn}')
PCAP_PATH = f'pcaps/{fn}'
NAME = os.path.splitext(fn)[0]
print('업로드 완료:', PCAP_PATH, '/ NAME =', NAME)

In [ ]:
# 7. 수집 → 압축 (Suricata/Zeek → evidence package + 드릴다운 번들)
!bash scripts/analyze.sh {PCAP_PATH}

In [ ]:
# 8. LLM 판정 (업로드한 pcap)
!python -u scripts/llm_analyze.py {NAME} \
    --base-url http://localhost:11434/v1 \
    --model qwen2.5:14b \
    --temperature 0.0

import json
print('\n=== verdict ===')
print(json.dumps(json.load(open(f'report/{NAME}.verdict.json')), ensure_ascii=False, indent=2))